In [0]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("MySparkApp") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

In [0]:
%sql
select * from samples.bakehouse.media_gold_reviews_chunked

In [0]:
df = spark.read.csv("/databricks-datasets/Rdatasets/data-001/csv/ggplot2/diamonds.csv", header=True)
# df.show()
display(df)
df.write.mode("overwrite").saveAsTable("mytable")

In [0]:
display(dbutils.fs.ls('/'))
display(dbutils.fs.ls('/databricks-datasets/'))

In [0]:
# can directly read tables as well
tableDF = spark.read.table('samples.accuweather.forecast_daily_calendar_imperial')
display(tableDF)

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField('user_id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('salary', DoubleType(), True)
])

df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv('/Volumes/workspace/default/mydata/salary.csv')
)

display(df)

In [0]:
# read 1st file consiting of header row
headerDF = (
    spark.read
    .option("header", "true")
    # .option("inferSchema", "true")
    .csv("/Volumes/samples/databricks/datasets/airlines/part-00000")
)
# display(headerDF)

# extract column names
columns = headerDF.columns

# fetch the list of files having no header row, exclude the first file
files = dbutils.fs.ls('/Volumes/samples/databricks/datasets/airlines/')
remaining = [f.path for f in files if f.name != 'part-00000']

# read all remaining files having no header row
dataDF = (
    spark.read
    .csv(remaining)
    .toDF(*columns)
)
# display(dataDF)

# union both dfs for complete data
finalDF = headerDF.unionByName(dataDF)
display(finalDF)




In [0]:
orders.select(
    col("customer_id"),
    col("product_id"),
    (col("qty") * col("price")).alias("total_amount")
)

In [0]:
# filtering - filter()
df.filter(condition)
df.where(condition)

# .filter() and .where() are same technically, just alias to each other

df.filter(c1 & c2)
# & -- and 
# | -- or
# ~ -- not

df.filter(c1.isNull())
df.filter(c1.isNotNull())

df.filter(c1.isin(list of options))

In [0]:
from pyspark.sql.functions import col, when

data = [
    (1,'Alice',50000, 'IT', True),
    (2,'Bob',80000, 'HR', True),
    (3,'Alice',50000, 'IT', True),
    (4,'David',120000, 'IT', False)
]

columns = ['id', 'name', 'salary', 'dept', 'is_active']

df = spark.createDataFrame(data, schema=columns)
display(df)

# 1. Create annual_salary from salary.
df = df.withColumn('annual_salary', col('salary') * 12)

# 2. Create salary_category:
# < 60000 → "Low"
# 60000–99999 → "Medium"
# >= 100000 → "High"
df = df.withColumn('salary_category', 
                   when(col('salary') < 60000, 'Low')
                   .when(col('salary') >= 100000, 'High')
                   .otherwise('Medium')
                )

# 3. Remove is_active.
df = df.drop('is_active')

# 4. Rename dept to department using alias().

# 5. Remove completely duplicate rows.
df = df.distinct()

# 6. Select only: id, name, department, annual_salary, salary_category
df = df.select('id', "name", col("dept").alias("department"), "annual_salary", "salary_category")

display(df)

In [0]:
from pyspark.sql.functions import lit, col, when

data = [
    (1, "Alice", "50000", "IT"),
    (2, "Bob", "80000", "HR"),
    (3, "Charlie", "120000", "IT"),
    (4, "David", "unknown", "Finance")
]

columns = ["id", "name", "salary", "department"]

df = spark.createDataFrame(data, columns)

display(df)

df = (
    df
    .withColumn('salary', col('salary').try_cast('integer'))
    .withColumn('annual_salary', col('salary') * 12)
    .withColumn('salary_category', 
                when(col('salary') < 60000, 'Low')
                .when(col('salary') >= 100000, 'High')
                .otherwise('Medium'))
    .withColumn('country', lit('India'))
    .withColumn('is_tech', 
                when(col('department').isin('IT', 'Engineering'), True)
                .otherwise(False))
    .filter(
        (col('salary') >= 80000) & (col('department').isin('IT', 'Engineering'))
    )
)

display(df)

In [0]:
from pyspark.sql.functions import col, coalesce, lit

data = [
    (1, "Alice", 50000, "alice@gmail.com"),
    (2, "Bob", None, "bob@gmail.com"),
    (3, "Charlie", 80000, None),
    (4, "David", None, None),
    (5, "Eve", 120000, "eve@gmail.com")
]

columns = ["id", "name", "salary", "email"]

df = spark.createDataFrame(data, columns)
display(df)

nullsal = df.filter(col('salary').isNull())
display(nullsal)

validemail = df.filter(col('email').isNotNull())
display(validemail)

newdf = df.fillna({'salary':0, 'email':'Unknown'})
display(newdf)

df = (
    df
    .withColumn('contact', coalesce(col('email'), lit('Not Available')))
    .dropna(subset=['salary'])
    )
display(df)

In [0]:
from pyspark.sql.functions import sum, avg, count, min, max, countDistinct

data = [
    (101, "Electronics", 500),
    (101, "Electronics", 700),
    (101, "Clothing", 300),
    (102, "Electronics", 900),
    (102, "Clothing", 200),
    (102, "Clothing", 400),
    (103, "Electronics", 1000),
    (103, "Electronics", None)
]

columns = ["customer_id", "category", "amount"]

df = spark.createDataFrame(data, columns)
display(df)

df.groupby("customer_id").agg(sum('amount').alias('total')).show()

df.groupby("customer_id").agg(avg('amount').alias('average')).show()

df.groupby("customer_id").agg(count('*').alias('transactions')).show()

df.groupby("customer_id").agg(count('amount').alias('amounts')).show()

df.groupby("customer_id", "category").agg(sum('amount').alias('category_total')).show()

df.groupBy("customer_id").agg(
    sum("amount").alias('total_amount'),
    avg("amount").alias('average_amount'),
    count('*').alias('transaction_count'),
    min("amount").alias('min_amount'),
    max("amount").alias('max_amount')
).show()

df.groupby("customer_id").agg(countDistinct("category").alias('distinct_cat')).show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc

data = [
    (101, "Alice", "Active", 50000, "2026-09-01 10:00:00", 1),
    (101, "Alice", "Active", 50000, "2026-09-01 10:00:00", 1),
    (101, "Alice", "Gold",   55000, "2026-09-10 10:00:00", 2),
    (101, "Alice", "Premium",60000, "2026-09-10 10:00:00", 3),
    (102, "Bob",   "Active", 80000, "2026-09-05 09:00:00", 4),
    (102, "Bob",   "Gold",   85000, "2026-09-15 09:00:00", 5),
    (103, "Charlie","Active",70000, "2026-09-03 12:00:00", 6)
]

columns = [
    "customer_id",
    "name",
    "status",
    "salary",
    "updated_at",
    "event_id"
]

df = spark.createDataFrame(data, columns)
display(df)

df = df.dropDuplicates()

window = Window.partitionBy('customer_id').orderBy(col('updated_at').desc(), col('event_id').desc())

df = (
    df.withColumn('rn', row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

display(df)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("PySparkJoins") \
    .getOrCreate()


# Customers DataFrame
customers_data = [
    (1, "A"),
    (2, "B"),
    (3, "C")
]

customers_columns = [
    "customer_id",
    "name"
]

customers = spark.createDataFrame(
    customers_data,
    customers_columns
)


# Orders DataFrame
orders_data = [
    (101, 1, 500),
    (102, 1, 700),
    (103, 2, 300),
    (104, 5, 900)
]

orders_columns = [
    "order_id",
    "customer_id",
    "amount"
]

orders = spark.createDataFrame(
    orders_data,
    orders_columns
)


customers.show()
orders.show()

# 6
customers.join(orders, "customer_id", "inner").show()

# 7
customers.join(orders, "customer_id", "left").show()

# 8
customers.join(orders, "customer_id", "left").filter((orders.customer_id).isNull()).show()
# or
customers.join(orders, "customer_id", "left_anti").show()

# 9
customers.join(orders, "customer_id", "full_outer").show()

# 10
customers.join(orders, "customer_id", "left_semi").show()

# 19
customers.join(orders, "customer_id").groupBy("customer_id").count().filter('count > 1').show()

# 20
# customer_profile.select('customer_id').groupBy('customer_id').count().filter('count > 1').show()